# Regresion supervisada para Offer_Salary

Notebook centrado unicamente en predecir la variable `Offer_Salary` a partir del archivo `cleanData/dataset_offer_Salary.csv`.

Modelos incluidos:
- Regresion lineal
- Ridge
- Lasso
- Arbol de regresion
- Random Forest
- Gradient Boosting

La comparacion final usa `MAE`, `RMSE` y `R2` sobre el mismo conjunto de prueba para todos los modelos.

## Objetivo

Construir un flujo reproducible para entrenar, comparar e interpretar modelos de regresion sobre el dataset de salario de oferta.

Cada algoritmo guarda sus graficas, su informe en Markdown y su modelo serializado en una carpeta propia dentro de `job/outputs_regresion/`.

In [ ]:
from pathlib import Path
import pickle
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor

SEED = 42
np.random.seed(SEED)
warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid')

ROOT = Path.cwd()
if ROOT.name != 'job' and (ROOT / 'job').exists():
    ROOT = ROOT / 'job'

DATA_PATH = ROOT / 'cleanData' / 'dataset_offer_Salary.csv'
OUTPUT_ROOT = ROOT / 'outputs_regresion'
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

ALGOS = {
    'linear_regression': 'Regresion lineal',
    'ridge': 'Ridge',
    'lasso': 'Lasso',
    'decision_tree': 'Arbol de regresion',
    'random_forest': 'Random Forest',
    'gradient_boosting': 'Gradient Boosting',
}

results = []

try:
    OHE = OneHotEncoder(handle_unknown='ignore', sparse_output=False)
except TypeError:
    OHE = OneHotEncoder(handle_unknown='ignore', sparse=False)

def make_directories(slug):
    algo_root = OUTPUT_ROOT / slug
    img_dir = algo_root / 'imagenes'
    report_dir = algo_root / 'informe'
    img_dir.mkdir(parents=True, exist_ok=True)
    report_dir.mkdir(parents=True, exist_ok=True)
    return algo_root, img_dir, report_dir

def evaluar_regresion(y_real, y_pred):
    mse = mean_squared_error(y_real, y_pred)
    return {
        'MAE': mean_absolute_error(y_real, y_pred),
        'RMSE': float(np.sqrt(mse)),
        'R2': r2_score(y_real, y_pred),
    }

def save_figure(fig, path):
    fig.savefig(path, dpi=160, bbox_inches='tight')
    plt.close(fig)

def residual_plots(y_true, y_pred, title_prefix, img_dir, slug):
    residuals = y_true - y_pred

    fig1, ax1 = plt.subplots(figsize=(7, 5))
    ax1.scatter(y_true, y_pred, alpha=0.35, color='#1f77b4')
    min_val = min(float(np.min(y_true)), float(np.min(y_pred)))
    max_val = max(float(np.max(y_true)), float(np.max(y_pred)))
    ax1.plot([min_val, max_val], [min_val, max_val], '--', color='black', linewidth=1)
    ax1.set_title(f'{title_prefix}: real vs predicho')
    ax1.set_xlabel('Valor real')
    ax1.set_ylabel('Valor predicho')
    save_figure(fig1, img_dir / f'{slug}_real_vs_predicho.png')

    fig2, ax2 = plt.subplots(figsize=(7, 5))
    ax2.hist(residuals, bins=30, color='#d62728', alpha=0.8)
    ax2.set_title(f'{title_prefix}: distribucion de residuos')
    ax2.set_xlabel('Residuo')
    ax2.set_ylabel('Frecuencia')
    save_figure(fig2, img_dir / f'{slug}_residuos_hist.png')

    fig3, ax3 = plt.subplots(figsize=(7, 5))
    ax3.scatter(y_pred, residuals, alpha=0.35, color='#2ca02c')
    ax3.axhline(0, color='black', linewidth=1)
    ax3.set_title(f'{title_prefix}: residuos vs prediccion')
    ax3.set_xlabel('Prediccion')
    ax3.set_ylabel('Residuo')
    save_figure(fig3, img_dir / f'{slug}_residuos_vs_prediccion.png')

    return residuals

def compact_params(params, keys):
    return {key: params[key] for key in keys if key in params}

def feature_table_from_model(pipeline):
    feature_names = pipeline.named_steps['preprocessor'].get_feature_names_out()
    model = pipeline.named_steps['model']
    if hasattr(model, 'coef_'):
        weights = np.ravel(model.coef_)
        weight_name = 'coeficiente'
    else:
        weights = np.ravel(model.feature_importances_)
        weight_name = 'importancia'
    df = pd.DataFrame({'feature': feature_names, weight_name: weights})
    df['abs_value'] = df[weight_name].abs()
    return df.sort_values('abs_value', ascending=False).reset_index(drop=True), weight_name

def plot_top_weights(pipeline, title, img_dir, slug):
    weight_df, weight_name = feature_table_from_model(pipeline)
    top_df = weight_df.head(15).sort_values('abs_value', ascending=True)
    fig, ax = plt.subplots(figsize=(10, 6))
    colors = ['#2ca02c' if value >= 0 else '#d62728' for value in top_df[weight_name]]
    ax.barh(top_df['feature'], top_df[weight_name], color=colors)
    ax.set_title(title)
    ax.set_xlabel(weight_name.capitalize())
    ax.set_ylabel('Feature')
    save_figure(fig, img_dir / f'{slug}_top_weights.png')
    return weight_df

def build_params_markdown(best_params):
    if not best_params:
        return '- Modelo base sin busqueda de hiperparametros.'
    lines = []
    for key, value in best_params.items():
        lines.append(f'- `{key}`: `{value}`')
    return '\n'.join(lines)

def build_top_features_markdown(weight_df, weight_name, model_kind, top_n=10):
    top_df = weight_df.head(top_n).copy()
    if model_kind == 'coef':
        top_pos = top_df[top_df[weight_name] > 0].head(5)
        top_neg = top_df[top_df[weight_name] < 0].head(5)
        lines = ['### Variables con mayor peso en el modelo', '']
        if not top_pos.empty:
            lines.append('**Coeficientes positivos (empujan salario hacia arriba):**')
            for _, row in top_pos.iterrows():
                lines.append(f"- `{row['feature']}`: `{row[weight_name]:.4f}`")
            lines.append('')
        if not top_neg.empty:
            lines.append('**Coeficientes negativos (empujan salario hacia abajo):**')
            for _, row in top_neg.iterrows():
                lines.append(f"- `{row['feature']}`: `{row[weight_name]:.4f}`")
            lines.append('')
        return '\n'.join(lines)

    lines = ['### Variables con mayor importancia', '']
    for _, row in top_df.iterrows():
        lines.append(f"- `{row['feature']}`: `{row[weight_name]:.4f}`")
    lines.append('')
    return '\n'.join(lines)

def write_report(report_path, slug, model_name, best_params, train_metrics, test_metrics, notes, weight_df, weight_name, model_kind):
    params_md = build_params_markdown(best_params)
    top_features_md = build_top_features_markdown(weight_df, weight_name, model_kind)
    overfit_gap = test_metrics['RMSE'] - train_metrics['RMSE']

    if model_kind == 'coef':
        model_read = 'Modelo lineal con interpretacion directa por coeficientes.'
    else:
        model_read = 'Modelo basado en arboles con lectura por importancia de variables.'

    content = f'''# Informe ejecutivo - {model_name}

## 1. Resumen ejecutivo

Este informe presenta el desempeño del modelo **{model_name}** para predecir `Offer_Salary` a partir de `dataset_offer_Salary.csv`.

{model_read}

### Resultado principal

- `MAE test`: {test_metrics['MAE']:.4f}
- `RMSE test`: {test_metrics['RMSE']:.4f}
- `R2 test`: {test_metrics['R2']:.4f}
- `RMSE train`: {train_metrics['RMSE']:.4f}
- `Brecha RMSE (test - train)`: {overfit_gap:.4f}

**Lectura operativa:** {notes}

## 2. Archivos entregables

- Notebook principal: [regresion_offer_salary_completo.ipynb](../../../regresion_offer_salary_completo.ipynb)
- Modelo serializado: `{slug}_offer_salary.pkl`
- Informe actual: `{slug}_offer_salary.md`

## 3. Configuracion final del modelo

{params_md}

## 4. Metricas de entrenamiento vs prueba

- **Train**: `MAE={train_metrics['MAE']:.4f}` | `RMSE={train_metrics['RMSE']:.4f}` | `R2={train_metrics['R2']:.4f}`
- **Test**: `MAE={test_metrics['MAE']:.4f}` | `RMSE={test_metrics['RMSE']:.4f}` | `R2={test_metrics['R2']:.4f}`

Interpretacion: una brecha train-test pequena sugiere mejor capacidad de generalizacion; una brecha amplia sugiere riesgo de sobreajuste.

## 5. Lectura ejecutiva de las graficas

### Real vs predicho

![Real vs predicho](../imagenes/{slug}_real_vs_predicho.png)

Si los puntos se acercan a la diagonal, el modelo predice salarios con mejor precision.

### Distribucion de residuos

![Distribucion de residuos](../imagenes/{slug}_residuos_hist.png)

Permite revisar si los errores se concentran cerca de cero o si hay colas con errores grandes.

### Residuos vs prediccion

![Residuos vs prediccion](../imagenes/{slug}_residuos_vs_prediccion.png)

Permite detectar patrones de sesgo: por ejemplo, si el modelo falla mas en rangos altos de salario.

### Variables mas influyentes

![Top variables](../imagenes/{slug}_top_weights.png)

{top_features_md}

## 6. Uso del modelo serializado

```python
import pickle
from pathlib import Path
import pandas as pd

artifact_path = Path('{slug}_offer_salary.pkl')
with open(artifact_path, 'rb') as f:
    artifact = pickle.load(f)

pipeline = artifact['pipeline']
feature_columns = artifact['feature_columns']

# Ejemplo: reemplazar por un registro real con las mismas columnas
nuevo = pd.DataFrame([{{col: 0 for col in feature_columns}}])
pred_salary = pipeline.predict(nuevo[feature_columns])[0]
print('Prediccion de salario:', round(float(pred_salary), 2))
```

## 7. Conclusion

El modelo **{model_name}** queda disponible para uso operativo y comparacion tecnica frente a los otros algoritmos del notebook. Este informe deja trazabilidad completa de metricas, parametros, variables relevantes y artefactos.
'''
    report_path.write_text(content, encoding='utf-8')

def run_model(slug, model_name, estimator, param_grid=None, notes='', model_kind='tree'):
    algo_root, img_dir, report_dir = make_directories(slug)
    pipeline = Pipeline([
        ('preprocessor', preprocessor),
        ('model', estimator),
    ])

    if param_grid is not None:
        search = GridSearchCV(
            pipeline,
            param_grid=param_grid,
            scoring='neg_root_mean_squared_error',
            cv=5,
            n_jobs=-1,
            refit=True,
        )
        search.fit(X_train, y_train)
        fitted = search.best_estimator_
        best_params = search.best_params_
    else:
        fitted = pipeline.fit(X_train, y_train)
        best_params = compact_params(estimator.get_params(), ['fit_intercept', 'positive', 'n_jobs', 'random_state'])

    pred_train = fitted.predict(X_train)
    pred_test = fitted.predict(X_test)
    train_metrics = evaluar_regresion(y_train, pred_train)
    test_metrics = evaluar_regresion(y_test, pred_test)

    residual_plots(y_test, pred_test, model_name, img_dir, slug)

    weight_df = plot_top_weights(fitted, f'{model_name}: top weights', img_dir, slug)

    model_path = report_dir / f'{slug}_offer_salary.pkl'
    payload = {
        'pipeline': fitted,
        'best_params': best_params,
        'train_metrics': train_metrics,
        'test_metrics': test_metrics,
        'feature_columns': X.columns.tolist(),
        'numeric_columns': numeric_cols,
        'categorical_columns': categorical_cols,
        'model_name': model_name,
    }
    with open(model_path, 'wb') as f:
        pickle.dump(payload, f)

    report_path = report_dir / f'{slug}_offer_salary.md'
    write_report(
        report_path=report_path,
        slug=slug,
        model_name=model_name,
        best_params=best_params,
        train_metrics=train_metrics,
        test_metrics=test_metrics,
        notes=notes,
        weight_df=weight_df,
        weight_name='coeficiente' if model_kind == 'coef' else 'importancia',
        model_kind=model_kind,
    )

    results.append({
        'modelo': model_name,
        'slug': slug,
        'MAE_train': train_metrics['MAE'],
        'RMSE_train': train_metrics['RMSE'],
        'R2_train': train_metrics['R2'],
        'MAE_test': test_metrics['MAE'],
        'RMSE_test': test_metrics['RMSE'],
        'R2_test': test_metrics['R2'],
        'best_params': best_params,
        'model_path': str(model_path),
        'report_path': str(report_path),
    })

    return fitted, train_metrics, test_metrics, best_params, weight_df

In [2]:
df = pd.read_csv(DATA_PATH)

if 'Offer_Salary' not in df.columns:
    raise ValueError('The target column Offer_Salary was not found in the dataset.')

display(df.head())
display(pd.DataFrame({
    'indicador': ['filas', 'columnas', 'nulos totales'],
    'valor': [df.shape[0], df.shape[1], int(df.isna().sum().sum())]
}))
display(df.describe(include='all').T.head(20))

X = df.drop(columns=['Offer_Salary'])
y = df['Offer_Salary']

numeric_cols = X.select_dtypes(include=['number']).columns.tolist()
categorical_cols = X.select_dtypes(exclude=['number']).columns.tolist()

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=SEED,
    shuffle=True,
)

preprocessor = ColumnTransformer([
    ('numeric', Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler', StandardScaler()),
    ]), numeric_cols),
    ('categorical', Pipeline([
        ('imputer', SimpleImputer(strategy='most_frequent')),
        ('onehot', OHE),
    ]), categorical_cols),
] )

print('Train shape:', X_train.shape)
print('Test shape:', X_test.shape)
print('Numerical columns:', len(numeric_cols))
print('Categorical columns:', len(categorical_cols))

,GPA,University_Rating,Major_Category,Region,Prior_Internships,Extra_Curricular_Activities,Networking_Events_Attended,School_Size,Primary_Search_Platform,Months_Searching,Applications_Submitted,First_Round_Interviews,Second_Round_Interviews,Offer_Salary
0,2.81,Mid-tier,Healthcare,West,3,2,6,Medium,LinkedIn,1,9,2,2,59785.0
1,3.01,Mid-tier,STEM,Northeast,0,3,2,Medium,Handshake,7,19,2,1,69910.0
2,3.22,Lower-tier,STEM,West,2,1,1,Medium,Indeed,6,67,2,1,73669.0
3,3.59,Top-tier,STEM,Midwest,2,1,3,Small,LinkedIn,5,31,2,2,83426.0
4,2.58,Mid-tier,Humanities,Midwest,3,2,1,Medium,LinkedIn,6,28,2,1,65189.0


,indicador,valor
0,filas,34229
1,columnas,14
2,nulos totales,0


,count,unique,top,freq,mean,std,min,25%,50%,75%,max
GPA,34229.0,NaN,NaN,NaN,3.244201,0.400699,2.0,2.97,3.25,3.53,4.0
University_Rating,34229,3,Mid-tier,16591,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Major_Category,34229,5,STEM,12121,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Region,34229,4,South,8644,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Prior_Internships,34229.0,NaN,NaN,NaN,1.543078,1.201039,0.0,1.0,1.0,2.0,5.0
Extra_Curricular_Activities,34229.0,NaN,NaN,NaN,2.014958,1.41461,0.0,1.0,2.0,3.0,10.0
Networking_Events_Attended,34229.0,NaN,NaN,NaN,3.000935,1.737104,0.0,2.0,3.0,4.0,12.0
School_Size,34229,3,Large,13680,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Primary_Search_Platform,34229,3,LinkedIn,14627,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Months_Searching,34229.0,NaN,NaN,NaN,6.73762,3.419668,1.0,4.0,7.0,10.0,12.0


Train shape: (27383, 13)
Test shape: (6846, 13)
Numerical columns: 8
Categorical columns: 5


## 1. Regresion lineal

Modelo base para tener una referencia interpretable y comparar el resto de algoritmos.

In [3]:
linear_model, linear_train, linear_test, linear_params, linear_weights = run_model(
    slug='linear_regression',
    model_name='Regresion lineal',
    estimator=LinearRegression(),
    param_grid=None,
    notes='La regresion lineal sirve como linea base interpretable. Los coeficientes positivos empujan la prediccion del salario hacia arriba y los negativos la reducen, pero esta lectura no implica causalidad.',
    model_kind='coef',
)

display(pd.DataFrame([{'conjunto': 'train', **linear_train}, {'conjunto': 'test', **linear_test}]))
display(linear_weights.head(15))

,conjunto,MAE,RMSE,R2
0,train,6388.027976,8014.240212,0.714007
1,test,6341.336190,7944.591527,0.708278


,feature,coeficiente,abs_value
0,categorical__Major_Category_STEM,14080.843331,14080.843331
1,categorical__Primary_Search_Platform_LinkedIn,6999.126163,6999.126163
2,categorical__University_Rating_Top-tier,6760.409105,6760.409105
3,categorical__Major_Category_Arts,-6287.900014,6287.900014
4,categorical__Major_Category_Humanities,-6092.398456,6092.398456
5,categorical__Primary_Search_Platform_Indeed,-6027.260050,6027.260050
6,categorical__Major_Category_Healthcare,-5782.673039,5782.673039
7,numeric__Prior_Internships,4811.739032,4811.739032
8,categorical__Major_Category_Business,4082.128178,4082.128178
9,categorical__University_Rating_Lower-tier,-3402.002065,3402.002065


## 2. Ridge y Lasso

Se prueban varios valores de `alpha` para controlar la complejidad del modelo y mejorar la generalizacion.

In [4]:
ridge_grid = {'model__alpha': [0.01, 0.1, 1, 10, 100]}
lasso_grid = {'model__alpha': [0.001, 0.01, 0.1, 1, 10]}

ridge_model, ridge_train, ridge_test, ridge_params, ridge_weights = run_model(
    slug='ridge',
    model_name='Ridge',
    estimator=Ridge(random_state=SEED),
    param_grid=ridge_grid,
    notes='Ridge penaliza coeficientes grandes con regularizacion L2. Es util cuando se busca estabilidad y se quiere evitar que una sola variable domine el ajuste.',
    model_kind='coef',
)

lasso_model, lasso_train, lasso_test, lasso_params, lasso_weights = run_model(
    slug='lasso',
    model_name='Lasso',
    estimator=Lasso(max_iter=20000, random_state=SEED),
    param_grid=lasso_grid,
    notes='Lasso aplica regularizacion L1 y puede llevar coeficientes a cero. Por eso sirve como seleccion automatica de variables y mantiene un modelo mas simple.',
    model_kind='coef',
)

display(pd.DataFrame([
    {'modelo': 'Ridge', 'conjunto': 'train', **ridge_train},
    {'modelo': 'Ridge', 'conjunto': 'test', **ridge_test},
    {'modelo': 'Lasso', 'conjunto': 'train', **lasso_train},
    {'modelo': 'Lasso', 'conjunto': 'test', **lasso_test},
]))

,modelo,conjunto,MAE,RMSE,R2
0,Ridge,train,6388.034450,8014.240424,0.714007
1,Ridge,test,6341.340304,7944.566869,0.708279
2,Lasso,train,6388.418863,8014.753861,0.713971
3,Lasso,test,6339.790720,7941.934774,0.708473


## 3. Arbol de regresion

Este modelo captura relaciones no lineales y ofrece una estructura mas interpretable por reglas de corte.

In [5]:
tree_grid = {
    'model__max_depth': [3, 5, 8, None],
    'model__min_samples_leaf': [1, 3, 5],
    'model__min_samples_split': [2, 5, 10],
}

tree_model, tree_train, tree_test, tree_params, tree_weights = run_model(
    slug='decision_tree',
    model_name='Arbol de regresion',
    estimator=DecisionTreeRegressor(random_state=SEED),
    param_grid=tree_grid,
    notes='El arbol de regresion modela relaciones no lineales mediante reglas de decision. Es interpretable, pero puede sobreajustar si la profundidad crece demasiado.',
    model_kind='importance',
)

display(pd.DataFrame([{'conjunto': 'train', **tree_train}, {'conjunto': 'test', **tree_test}]))

,conjunto,MAE,RMSE,R2
0,train,6322.253459,7930.404596,0.719959
1,test,6510.068935,8152.485547,0.692810


In [6]:
rf_grid = {
    'model__n_estimators': [200, 400],
    'model__max_depth': [None, 8, 15],
    'model__min_samples_leaf': [1, 3, 5],
}

rf_model, rf_train, rf_test, rf_params, rf_weights = run_model(
    slug='random_forest',
    model_name='Random Forest',
    estimator=RandomForestRegressor(random_state=SEED, n_jobs=-1),
    param_grid=rf_grid,
    notes='Random Forest promedia muchos arboles para reducir varianza y suele generalizar mejor que un arbol unico. Es una buena alternativa cuando se busca rendimiento robusto.',
    model_kind='importance',
)

display(pd.DataFrame([{'conjunto': 'train', **rf_train}, {'conjunto': 'test', **rf_test}]))

,conjunto,MAE,RMSE,R2
0,train,6233.935374,7820.863972,0.727642
1,test,6421.544429,8031.734613,0.701843


In [7]:
gb_grid = {
    'model__n_estimators': [100, 200],
    'model__learning_rate': [0.03, 0.05, 0.1],
    'model__max_depth': [2, 3],
    'model__subsample': [0.8, 1.0],
}

gb_model, gb_train, gb_test, gb_params, gb_weights = run_model(
    slug='gradient_boosting',
    model_name='Gradient Boosting',
    estimator=GradientBoostingRegressor(random_state=SEED),
    param_grid=gb_grid,
    notes='Gradient Boosting construye arboles secuenciales corrigiendo errores del anterior. Suele dar muy buen balance entre sesgo y varianza cuando se ajusta con una grilla pequena y controlada.',
    model_kind='importance',
)

display(pd.DataFrame([{'conjunto': 'train', **gb_train}, {'conjunto': 'test', **gb_test}]))

,conjunto,MAE,RMSE,R2
0,train,6362.605433,7981.482028,0.716340
1,test,6349.863085,7951.833873,0.707745


In [8]:
comparison_df = pd.DataFrame(results).sort_values('RMSE_test').reset_index(drop=True)
comparison_path = OUTPUT_ROOT / 'comparacion_modelos_regresion.csv'
comparison_df.to_csv(comparison_path, index=False)

display(comparison_df[['modelo', 'MAE_test', 'RMSE_test', 'R2_test', 'best_params']])

fig, ax = plt.subplots(figsize=(11, 5))
ax.bar(comparison_df['modelo'], comparison_df['RMSE_test'], color='#1f77b4')
ax.set_title('Comparacion de RMSE en test')
ax.set_ylabel('RMSE test')
ax.tick_params(axis='x', rotation=25)
save_figure(fig, OUTPUT_ROOT / 'comparacion_rmse_test.png')

fig2, ax2 = plt.subplots(figsize=(11, 5))
ax2.bar(comparison_df['modelo'], comparison_df['R2_test'], color='#2ca02c')
ax2.set_title('Comparacion de R2 en test')
ax2.set_ylabel('R2 test')
ax2.tick_params(axis='x', rotation=25)
save_figure(fig2, OUTPUT_ROOT / 'comparacion_r2_test.png')

best_model_row = comparison_df.iloc[0]
print('Mejor modelo por RMSE test:', best_model_row['modelo'])
print('RMSE test:', round(best_model_row['RMSE_test'], 4))
print('R2 test:', round(best_model_row['R2_test'], 4))

,modelo,MAE_test,RMSE_test,R2_test,best_params
0,Lasso,6339.790720,7941.934774,0.708473,{'model__alpha': 10}
1,Ridge,6341.340304,7944.566869,0.708279,{'model__alpha': 1}
2,Regresion lineal,6341.336190,7944.591527,0.708278,"{'fit_intercept': True, 'positive': False, 'n_..."
3,Gradient Boosting,6349.863085,7951.833873,0.707745,"{'model__learning_rate': 0.1, 'model__max_dept..."
4,Random Forest,6421.544429,8031.734613,0.701843,"{'model__max_depth': 8, 'model__min_samples_le..."
5,Arbol de regresion,6510.068935,8152.485547,0.692810,"{'model__max_depth': 8, 'model__min_samples_le..."


Mejor modelo por RMSE test: Lasso
RMSE test: 7941.9348
R2 test: 0.7085


## Cierre

El notebook queda listo como base de trabajo para el taller de regresion sobre `Offer_Salary`.

Cada modelo ya deja su estructura de salidas en `job/outputs_regresion/`, con `imagenes/`, `informe/` y el archivo `.pkl` correspondiente.